# Catalog queries with DuckDB

**Environment:** [uv](https://docs.astral.sh/uv/) project venv from the repo README (`uv venv --python 3.11 .venv`, `uv sync --extra desi --extra dev`). In Jupyter, use kernel **Python (data-lake)** (`uv run python -m ipykernel install --user --name=data-lake --display-name "Python (data-lake)"`).

This notebook demonstrates how to query the HATS-partitioned Parquet catalogs using `CatalogAccessor`.
All queries run in-process via DuckDB – no server required.

**§8** — homogenizing photometry (WISE Vega → AB). **§9** — fast retrieval from a **master association** table for a list of object IDs (cross-match by position; catalogs may use different HEALPix orders). **§10** — lake-wide column discovery (`dl-describe-lake` / `dl-describe-master` / `dl-describe-survey`) before writing joins.

In [ ]:
from pathlib import Path
from data_lake.io.catalog import CatalogAccessor, MultiCatalogAccessor
from data_lake.io.crossmatch import CrossmatchAccessor

LAKE_ROOT = Path("/data/lake")  # adjust to your lake root
SURVEY = "des_dr2"

## 1. Open a catalog and inspect its schema

In [ ]:
cat = CatalogAccessor(LAKE_ROOT, SURVEY)
print(cat)
print(f"\nTotal columns: {len(cat.columns)}")
print("First 20 columns:", cat.columns[:20])

## 2. Arbitrary SQL query (column projection)

DuckDB applies column projection natively – only the requested columns are read from disk.

In [ ]:
df = cat.query("""
    SELECT source_id, ra, dec, mag_i, mag_r
    FROM catalog
    WHERE mag_i < 22.0
      AND mag_i - mag_r BETWEEN 0.2 AND 0.8
    ORDER BY mag_i
    LIMIT 1000
""")
df.head()

## 3. Cone search

In [ ]:
# Search within 0.5 deg of a field centre
cone = cat.sources_in_cone(
    ra=150.0,
    dec=2.2,
    radius_deg=0.5,
    columns=["source_id", "ra", "dec", "mag_i"],
)
print(f"Sources in cone: {len(cone)}")
cone.head()

## 4. Return as Astropy Table (for astronomers preferring familiar APIs)

In [ ]:
astropy_table = cat.sources_in_cone(
    ra=150.0, dec=2.2, radius_deg=0.5, fmt="astropy"
)
astropy_table[:5]

## 5. Multi-survey join via shared DuckDB connection

In [ ]:
with MultiCatalogAccessor(LAKE_ROOT, ["des_dr2", "kids_dr4"]) as multi:
    joined = multi.query("""
        SELECT a.source_id AS sid_des,
               b.source_id AS sid_kids,
               a.mag_i    AS mag_i_des,
               b.mag_i    AS mag_i_kids
        FROM des_dr2 AS a
        JOIN kids_dr4 AS b
          ON a.source_id = b.des_source_id   -- assumes crossmatch already applied
        WHERE a.mag_i < 22.0
        LIMIT 500
    """)
joined.head()

## 6. Cross-match query

In [ ]:
with CrossmatchAccessor(LAKE_ROOT, "des_dr2", "kids_dr4") as xm:
    matches = xm.get_matches(source_id_a=12345678, max_sep_arcsec=1.0)
matches

## 7. Tile-level access (retrieve all sources in a HEALPix tile)

In [ ]:
tile_df = cat.sources_in_tile(npix=1234, columns=["source_id", "ra", "dec", "mag_i"])
print(f"Sources in tile 1234: {len(tile_df)}")
tile_df.head()

## 8. Homogenizing photometry (example: WISE Vega → AB)

Archival catalogs arrive in **native units and column names** (Gaia: mmag and `phot_g_mean_mag`; DES: nanomaggy; WISE/AllWISE: **Vega magnitudes** in `w1mpro` … `w4mpro`). Ingest (`dl-ingest-catalog*`) should preserve source columns; **homogenization** is a separate ETL layer you can apply:

| When | Pros | Cons |
|------|------|------|
| **At query time** (DuckDB / `CatalogAccessor`) | No re-ingest; easy to iterate on definitions | Recomputed every query; not stored in Parquet |
| **At ingest time** (derived columns in FITS/CSV before ingest) | Uniform lake for ML and cross-survey joins | Must re-ingest or add columns to staging files |

Converting Gaia CSV → FITS speeds **read/ingest** only; it does not change flux units. This section shows **query-time** conversion for AllWISE using official WISE offsets ([WISE calibration §4.4.h](https://wise2.ipac.caltech.edu/docs/release/allsky/expsup/sec4_4h.html)).

**AllWISE Vega → AB offsets** (Δm such that **m_AB = m_Vega + Δm**):

| Band | Δm (Vega → AB) |
|------|----------------|
| W1 | 2.699 |
| W2 | 3.339 |
| W3 | 5.174 |
| W4 | 6.620 |

In [ ]:
# WISE AllWISE photometry: Vega system → AB (query-time homogenization)
# Offsets from WISE Explanatory Supplement Table 3 (sec 4.4.h)
WISE_VEGA_TO_AB = {
    "w1": 2.699,
    "w2": 3.339,
    "w3": 5.174,
    "w4": 6.620,
}

LAKE_ROOT = Path("/data/lake")  # adjust
WISE_SURVEY = "ALLWISE"          # must match catalogs/<name>/ on disk

wise_root = LAKE_ROOT / "catalogs" / WISE_SURVEY
if not wise_root.exists():
    raise FileNotFoundError(
        f"No catalog at {wise_root}. Ingest AllWISE first or set WISE_SURVEY / LAKE_ROOT."
    )

wise_cat = CatalogAccessor(LAKE_ROOT, WISE_SURVEY)
print(wise_cat)
col_lower = {c.lower(): c for c in wise_cat.columns}
print("W1 Vega column:", col_lower.get("w1mpro", "(missing — check ingest column list)"))

### SQL: native Vega mags + homogenized AB columns

DuckDB evaluates expressions over the Parquet scan — only referenced columns are read. Filter invalid AllWISE magnitudes (non-detections / sentinels) before converting.

Native columns are **`w1mpro` … `w4mpro`** (Vega). We add **`w1_ab` … `w4_ab`** for analysis that expects AB (e.g. comparison to SDSS/DES `mag_*` or Gaia `phot_*` after their own conversions).

In [ ]:
d_w1, d_w2, d_w3, d_w4 = (
    WISE_VEGA_TO_AB["w1"],
    WISE_VEGA_TO_AB["w2"],
    WISE_VEGA_TO_AB["w3"],
    WISE_VEGA_TO_AB["w4"],
)

wise_ab = wise_cat.query(f"""
    SELECT
        source_id,
        ra,
        dec,
        w1mpro,
        w1mpro + {d_w1} AS w1_ab,
        w2mpro,
        w2mpro + {d_w2} AS w2_ab,
        w3mpro,
        w3mpro + {d_w3} AS w3_ab,
        w4mpro,
        w4mpro + {d_w4} AS w4_ab,
        -- WISE color in AB (same as Vega: AB offsets are per band)
        (w1mpro + {d_w1}) - (w2mpro + {d_w2}) AS w1_w2_ab
    FROM catalog
    WHERE w1mpro IS NOT NULL
      AND w2mpro IS NOT NULL
      AND w1mpro > 0 AND w1mpro < 90
      AND w2mpro > 0 AND w2mpro < 90
    LIMIT 5000
""")

wise_ab.head()

### Optional: Vega flux density → AB flux (Jy)

If you need **flux** rather than magnitude, use the WISE zero-magnitude flux densities *F*₀ (Jy) for a flat *F*<sub>ν</sub> spectrum (Table 1, same doc) and

\[
F_\mathrm{Vega} = F_0 \times 10^{-0.4\,m_\mathrm{Vega}}, \qquad
F_\mathrm{AB} = F_\mathrm{Vega} \times 10^{-0.4\,\Delta m}.
\]

Example for W1 only (`F0_W1 = 309.540` Jy):

In [ ]:
F0_W1_JY = 309.540
dm_w1 = WISE_VEGA_TO_AB["w1"]

wise_flux = wise_cat.query(f"""
    SELECT
        source_id,
        w1mpro,
        w1mpro + {dm_w1} AS w1_ab,
        {F0_W1_JY} * pow(10, -0.4 * w1mpro) AS flux_w1_vega_jy,
        {F0_W1_JY} * pow(10, -0.4 * w1mpro) * pow(10, -0.4 * {dm_w1}) AS flux_w1_ab_jy
    FROM catalog
    WHERE w1mpro IS NOT NULL AND w1mpro > 0 AND w1mpro < 90
    LIMIT 1000
""")
wise_flux.head()

## 9. Master association table + ID list (fast column retrieval)

Cross-matching is **positional** (STILTS, `build_crossmatch`, etc.). Partner surveys do **not** need the same `--norder` — only native **object IDs** and optional per-survey `norder` / `npix` for Zarr paths matter at query time.

Workflow:

1. Build a flat **master** Parquet (e.g. `associations/master_desi_euclid.parquet`) with your primary IDs and partner IDs (`desi_targetid`, `euclid_source_id`, `sep_arcsec`, …).
2. Register a **`want`** table of IDs you care about (CSV, Parquet, or a Python list).
3. **Join** `want → master → catalog` in DuckDB; list only the columns you need (column projection).

See also `examples/cross_survey_lsst_desi_euclid/query.sql` and README § “Fast retrieval with DuckDB”.

In [ ]:
import polars as pl
from IPython.display import display

# Paths — adjust to your deployment (no hardcoded site paths in the repo)
LAKE_ROOT = Path("/data/lake")
MASTER_PARQUET = LAKE_ROOT / "associations" / "master_desi_euclid.parquet"
HOME_SURVEY = "DESI_DR1"
PARTNER_SURVEY = "EUCLID_DR1"

# Example ID list (replace with your sample: CSV column, query result, etc.)
want_ids = [1000001, 1000002, 1000003]
want_df = pl.DataFrame({"id": want_ids})

if not MASTER_PARQUET.is_file():
    print(f"Skip SQL demo: master not found at {MASTER_PARQUET}")
    print("Build with STILTS or export matches to Parquet, then re-run this cell.")
else:
    home = CatalogAccessor(LAKE_ROOT, HOME_SURVEY)
    sid_home = home.source_id_column
    print(f"Home survey {HOME_SURVEY!r}, ID column {sid_home!r}")

### DuckDB: `want` → master → home + partner catalogs

`MultiCatalogAccessor` registers each survey as a view. Register `want` and `master`, then join on **native ID columns** (here `TARGETID` / `SOURCE_ID` — use names from `catalog_info.json` / `CatalogAccessor.columns`).

In [ ]:
if MASTER_PARQUET.is_file():
    home = CatalogAccessor(LAKE_ROOT, HOME_SURVEY)
    sid_home = home.source_id_column
    with MultiCatalogAccessor(LAKE_ROOT, [HOME_SURVEY, PARTNER_SURVEY]) as multi:
        multi._con.register("want", want_df)
        multi._con.execute(
            "CREATE OR REPLACE VIEW master AS SELECT * FROM read_parquet(?)",
            [str(MASTER_PARQUET)],
        )
        # Adjust master column names (desi_targetid, euclid_source_id) to your Parquet schema
        result = multi.query(f"""
            SELECT
                w.id AS home_id,
                m.euclid_source_id,
                m.sep_arcsec,
                d.Z,
                d.MAG_G,
                d.MAG_R,
                e.SOURCE_ID AS euclid_id
            FROM want AS w
            INNER JOIN master AS m ON m.desi_targetid = w.id
            INNER JOIN {HOME_SURVEY} AS d ON d.{sid_home} = w.id
            INNER JOIN {PARTNER_SURVEY} AS e ON e.SOURCE_ID = m.euclid_source_id
        """)
    display(result.head())
else:
    print("(no master file — see cell above)")

### Python API: batched lookup on one catalog

For a single survey, `get_sources_by_id` issues batched `IN (...)` queries (default chunk size 10 000). Use after you have partner IDs from the master, or when the home catalog is the only table you need.

In [ ]:
if MASTER_PARQUET.is_file():
    home = CatalogAccessor(LAKE_ROOT, HOME_SURVEY)
    sid_home = home.source_id_column
    # Columns must exist in the survey schema — adjust as needed
    cols = [c for c in [sid_home, "Z", "MAG_G", "MAG_R", "MAG_Z"] if c in home.columns]
    rows = home.get_sources_by_id(want_ids, columns=cols)
    display(rows.head())
else:
    print("(no master file)")

### Optional: tile-targeted read (when master stores `npix` per survey)

If the master includes `desi_norder` and `desi_npix` (at **that** survey's ingest order), you can read one `Npix=*.parquet` tile instead of scanning the full glob — useful for very large catalogs.

```sql
-- Dir = (npix // 10000) * 10000  (nested HEALPix directory convention)
SELECT d.TARGETID, d.Z
FROM want w
JOIN master m ON m.desi_targetid = w.id
JOIN read_parquet(
  'catalogs/DESI_DR1/Norder=' || m.desi_norder::VARCHAR
  || '/Dir=' || ((m.desi_npix // 10000) * 10000)::VARCHAR
  || '/Npix=' || m.desi_npix::VARCHAR || '.parquet'
) AS d ON d.TARGETID = w.id;
```

## 10. Column discovery (master → per-survey manifests)

Before building SQL in §9, discover **join keys** and **science columns** from on-disk manifests (no need to open every Parquet tile).

1. **`dl-refresh-lake-registry`** — writes `shared/registry/surveys.parquet` (catalog + spectra + cutout rows).
2. **`dl-describe-master`** — maps master ID columns → catalog `source_id` columns (`<master>.meta.json` sidecar).
3. **`dl-describe-survey <name>`** — column names, dtypes, roles; optional `--modality spectra|cutout`.
4. Optional overlays under `shared/registry/overlays/` (units, AB offsets) merge at describe time.

Then use §9: register `want`, join `master`, project only the columns you picked in step 3.

In [ ]:
import subprocess
from pathlib import Path

# LAKE_ROOT from §9 / setup cell — adjust to your deployment
lake = Path(LAKE_ROOT) if "LAKE_ROOT" in dir() else Path("/data/lake")

def _run(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)

_run(["dl-refresh-lake-registry", str(lake)])
_run(["dl-describe-lake", str(lake)])

master = lake / "associations" / "master_desi_euclid.parquet"
if master.is_file():
    _run(["dl-describe-master", str(master), str(lake)])
else:
    print(f"No master at {master} — skip dl-describe-master")

for survey in [HOME_SURVEY, PARTNER_SURVEY] if "HOME_SURVEY" in dir() else []:
    _run(["dl-describe-survey", survey, str(lake), "--limit", "12"])

# P3: generate want → master → catalog SQL from column picks + master.meta.json
if MASTER_PARQUET.is_file() and "HOME_SURVEY" in dir():
    from data_lake.query_from_master import build_select_from_master

    plan = build_select_from_master(
        lake,
        MASTER_PARQUET,
        HOME_SURVEY,
        {
            HOME_SURVEY: ["Z", "MAG_G", "MAG_R"],
            PARTNER_SURVEY: ["SOURCE_ID"],
        },
        master_columns=["euclid_source_id", "sep_arcsec"],
    )
    print(plan.all_sql())